# Лабораторная работа № 5

### Работа с нейронными сетями в Keras

Подключаем необходимые библиотеки:
- tensorflow - открытая библиотека для машинного обучения от Google для создания нейросетевых моделей
- pyplot потребуется для отрисовки картинок и графиков
- randint - генератор случайных чисел для выбора примеров из набора данных
- numpy потребуется для обработки массивов чисел

Кроме того, нам потребуется высокоуровневый фреймворк keras, который позволяет быстро и легко создавать нейронные сети из различных слоев

Недостающие библиотеки устанавливаются обычным образом, при помощи pip. Обратите внимание, что в зависимости от установленной версии tensorflow, вам может потребоваться специфическая версия keras. Если возникает ошибка, можно обнавить версию tensorflow до последней или установить необходимую версию keras. Например: pip install keras==2.11.0

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from random import randint
import numpy as np

Для примера используем стандартный набор данных MNIST (Modified National Institute of Standards and Technology database), содержащий изображения цифр, написанных от руки. 

Набор данных не обязательно скачивать вручную, его можно загрузить прямо из библиотеки:

In [ ]:
mnist = tf.keras.datasets.mnist

Заполним массивы для обучения и проверки при помощи mnist.load_data()

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

Посмотрим на объем данных:

In [ ]:
x_train.shape, y_train.shape

Массив x_train содержит 60 000 монохромных картинок 28х28 пикселей. Уменьшим 

Посмотрим на примеры изображений. Для отображения в оттенках серого используем соответствующую цветовую карту:

In [ ]:
plt.axis('off')
plt.imshow(x_test[randint(0, 10000)], cmap=plt.get_cmap('gray'))
plt.show()

Пришло время создать модель:

In [ ]:
model = tf.keras.models.Sequential()

Наша модель пока пустая и не содержит слоев с нейронами. Последовательные модели (Sequential) создаются послойно, слои могут быть созданы сразу при создании модели, или добавлены в имеющуются модель. Воспользуемся вторым способом:

In [ ]:
model.add(tf.keras.layers.Flatten(input_shape=(28, 28)))
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dropout(0.2))
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(10, activation='softmax'))

Мы добавили в модель несколько слоёв:
- Входной слой обычно имеет тип Input, но мы используем слой Flatten, который обычно используется после сверточных слоев. Этот слой может принять двумерный массив и преобразовать его в одномерный. Таким образом мы экономим операцию reshape для наших изображений.
- Первый скрытый слой является полносвязанным и содержит 128 нейронов, активируемых функцией ReLu
- Второй скрытый слой - слой регуляризации. Приведен для примера, с некоторой долей верятности в нем обрываются входные связи, что позволяет избежать переобучения
- Третий скрытый слой аналогичен первому скрытому
- Выходной слой содержит 10 нейронов для 10 классов - 10 цифр. Активируется функцией softmax.

Пришло время настроить нашу модель. Здесь мы задаем параметры оптимизации, функции потерь, метрики, а также (при необходимости) другие настройки.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

Пришло время обучить модель. Попробуем 5 эпох. Отладочную инфомацию сохраним в объекте history:

In [ ]:
history = model.fit(x_train, y_train, epochs=5)

Сохраним массивы точности и потерь по эпохам:

In [ ]:
train_loss = history.history['loss']
train_acc = history.history['accuracy']

Построим графики обучения модели:

In [ ]:
plt.plot(train_loss, label='train_loss')
plt.plot(train_acc, label='train_acc')
plt.xlabel('Epochs')
plt.ylabel('Acc-Loss')
plt.legend()
plt.show()

Проверим модель на тестовом наборе данных:

In [ ]:
model.evaluate(x_test,  y_test, verbose=2)

Сохраним массив предсказанных значений:

In [ ]:
result = model.predict(x_test)

Попробуем визуализировать результат нашей работы. 

In [ ]:
#Количество картинок по Х и Y
num_rows = 2
num_cols = 4
num_images = num_rows * num_cols
#Создадим объект figure
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
#А картинки нарисуем в нем субплотами
for i in range(num_images):
  #Случайный элемент массива
  num=randint(0, 10000)
  plt.subplot(num_rows, 2*num_cols, 2*i + 1)
  #Отключить шкалы осей
  plt.xticks([])
  plt.yticks([])
  #Рисуем картинку
  plt.imshow(x_test[num], cmap=plt.get_cmap('gray'))
  #Подпись к ней берем из массива, который вернула модель
  plt.xlabel(np.argmax(result[num]))

#### Дополнительно

Сохраним модель в h5 формате и визуализируем ее при помощи библиотеки netron:

In [ ]:
tf.keras.models.save_model(model, 'model.h5')

In [ ]:
import netron
netron.start('model.h5')

# Задания:

1. Выделите в датасете тестовый (test), тренировочный (train) и проверочный (val) наборы.
1. Создайте модель со своей архитектурой сети. Обясните выбор количества и параметров слоев.
2. Обучите модель на сформированном наборе данных
3. Изменяя гиперпараметры, число эпох обучения, архитектуру модели, метрики проверки качества добейтесь максимальной точности классификации. Опишите процесс своих поисков
4. Напишите общие выводы по работе. Подкрепите их графиками.
6* Попробуйте кросс-валидацию, опишите ее влияние на результат обучения